# XGBoost/RandomForest Model From CSV

This notebook retrains and compares two admission models using data from `data/tunisie_orientation_complete.csv`, then saves the selected best model to `backend/orientation/ml/model_main.joblib`.

The final model keeps the same feature contract used by the API:

- `score`
- `section_encoded`
- `filiere_id`


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
# Project paths
BACKEND_DIR = Path.cwd().resolve().parent
PROJECT_ROOT = BACKEND_DIR.parent
CSV_PATH = PROJECT_ROOT / 'data' / 'tunisie_orientation_complete.csv'
MODEL_PATH = BACKEND_DIR / 'orientation' / 'ml' / 'model_main.joblib'

print('Backend dir :', BACKEND_DIR)
print('Project root:', PROJECT_ROOT)
print('CSV path    :', CSV_PATH)
print('Model path  :', MODEL_PATH)
assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'

Backend dir : C:\Users\dali\EduStat-TN\backend
Project root: C:\Users\dali\EduStat-TN
CSV path    : C:\Users\dali\EduStat-TN\data\tunisie_orientation_complete.csv
Model path  : C:\Users\dali\EduStat-TN\backend\orientation\ml\model_main.joblib


In [3]:
# Load Django context to map filiere code -> filiere_id used by API model input
import sys

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'edustat.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

import django
django.setup()

from orientation.models import Filiere

code_to_id = {str(f.code): int(f.id) for f in Filiere.objects.all()}
print('Mapped filieres from DB:', len(code_to_id))

Mapped filieres from DB: 690


In [4]:
# Read CSV and prepare long-format yearly score thresholds
df = pd.read_csv(CSV_PATH)
df['Code_Filiere'] = df['Code_Filiere'].astype(str)
df['filiere_id'] = df['Code_Filiere'].map(code_to_id)

# Keep rows that can be mapped to DB filiere IDs
df = df[df['filiere_id'].notna()].copy()
df['filiere_id'] = df['filiere_id'].astype(int)

SECTION_ENCODING = {'M': 0, 'S': 1, 'T': 2, 'E': 3, 'L': 4, 'I': 5, 'SP': 6}
df['section_encoded'] = df['Section_Bac'].astype(str).str.upper().map(SECTION_ENCODING).fillna(0).astype(int)

score_cols = ['Score_2022', 'Score_2023', 'Score_2024', 'Score_2025']
for c in score_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

long_df = df.melt(
    id_vars=['Code_Filiere', 'filiere_id', 'Section_Bac', 'section_encoded'],
    value_vars=score_cols,
    var_name='year_col',
    value_name='threshold_score'
)
long_df['threshold_score'] = long_df['threshold_score'].astype(float)
long_df = long_df.dropna(subset=['threshold_score'])

print('Base rows:', len(df))
print('Long rows:', len(long_df))
long_df.head()

Base rows: 2880
Long rows: 10638


,Code_Filiere,filiere_id,Section_Bac,section_encoded,year_col,threshold_score
0,10101,691,L,4,Score_2022,98.1618
1,10101,691,S,1,Score_2022,113.8213
2,10101,691,I,5,Score_2022,113.3344
3,10101,691,E,3,Score_2022,97.8900
4,10102,692,L,4,Score_2022,109.9199


In [5]:
# Build stronger synthetic candidate samples around each yearly threshold
# Goal: improve separability while keeping API feature contract (score, section_encoded, filiere_id)
rng = np.random.default_rng(42)
samples = []

for _, row in long_df.iterrows():
    t = float(row['threshold_score'])
    fid = int(row['filiere_id'])
    sec = int(row['section_encoded'])

    # Positive samples: mostly above threshold, with a few near-threshold cases
    pos_far = t + rng.uniform(1.5, 12.0, size=5)
    pos_near = t + rng.uniform(0.2, 1.5, size=2)

    # Negative samples: mostly below threshold, with a few near-threshold cases
    neg_far = np.maximum(0.0, t - rng.uniform(1.5, 12.0, size=5))
    neg_near = np.maximum(0.0, t - rng.uniform(0.2, 1.5, size=2))

    for s in pos_far:
        samples.append((float(s), sec, fid, 1))
    for s in pos_near:
        samples.append((float(s), sec, fid, 1))
    for s in neg_far:
        samples.append((float(s), sec, fid, 0))
    for s in neg_near:
        samples.append((float(s), sec, fid, 0))

train_df = pd.DataFrame(samples, columns=['score', 'section_encoded', 'filiere_id', 'admitted'])

# Clamp extreme values to a realistic interval for this dataset
train_df['score'] = train_df['score'].clip(lower=0, upper=200)

print('Training rows:', len(train_df))
print(train_df['admitted'].value_counts(normalize=True).rename('class_ratio'))
train_df.head()

Training rows: 148932
admitted
1    0.5
0    0.5
Name: class_ratio, dtype: float64


,score,section_encoded,filiere_id,admitted
0,107.788339,4,691,1
1,104.270024,4,691,1
2,108.677078,4,691,1
3,106.984164,4,691,1
4,100.650662,4,691,1


In [6]:
# Tune and compare two models: XGBoost vs Random Forest (accuracy-oriented)
X = train_df[['score', 'section_encoded', 'filiere_id']]
y = train_df['admitted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_spaces = {
    'xgboost': {
        'estimator': XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=42,
            tree_method='hist',
            n_jobs=-1
        ),
        'params': {
            'n_estimators': [300, 500, 700, 900],
            'max_depth': [3, 4, 5, 6, 8],
            'learning_rate': [0.02, 0.03, 0.05, 0.08],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0],
            'min_child_weight': [1, 3, 5, 7],
            'reg_alpha': [0.0, 0.1, 0.3],
            'reg_lambda': [1.0, 1.5, 2.0, 3.0]
        },
        'n_iter': 24
    },
    'random_forest': {
        'estimator': RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ),
        'params': {
            'n_estimators': [300, 500, 700, 900],
            'max_depth': [10, 14, 18, 24, None],
            'min_samples_split': [2, 4, 8],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced']
        },
        'n_iter': 24
    }
}

results = []
trained_models = {}

for name, cfg in search_spaces.items():
    print(f'\n=== Tuning {name.upper()} ===')

    search = RandomizedSearchCV(
        estimator=cfg['estimator'],
        param_distributions=cfg['params'],
        n_iter=cfg['n_iter'],
        scoring='accuracy',
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)
    best_estimator = search.best_estimator_

    pred = best_estimator.predict(X_test)
    proba = best_estimator.predict_proba(X_test)[:, 1]

    model_metrics = {
        'model': name,
        'cv_accuracy_best': float(search.best_score_),
        'accuracy': float(accuracy_score(y_test, pred)),
        'auc_roc': float(roc_auc_score(y_test, proba)),
        'train_size': int(len(X_train)),
        'test_size': int(len(X_test)),
    }

    print('Best params:', search.best_params_)
    print('Metrics:', model_metrics)
    print('Classification report:')
    print(classification_report(y_test, pred, digits=4))

    results.append({**model_metrics, 'best_params': search.best_params_})
    trained_models[name] = best_estimator

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'best_params'} for r in results])
results_df = results_df.sort_values(by=['accuracy', 'cv_accuracy_best', 'auc_roc'], ascending=False)

print('\nModel ranking (best first):')
print(results_df[['model', 'cv_accuracy_best', 'accuracy', 'auc_roc']].to_string(index=False))

best_model_name = results_df.iloc[0]['model']
best_model = trained_models[best_model_name]
metrics = results_df.iloc[0].to_dict()
best_params = next(r['best_params'] for r in results if r['model'] == best_model_name)

print(f'\nSelected model: {best_model_name}')
print('Selected best params:', best_params)


=== Tuning XGBOOST ===
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'n_estimators': 500, 'min_child_weight': 7, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
Metrics: {'model': 'xgboost', 'cv_accuracy_best': 0.6549750304251123, 'accuracy': 0.6603551885050526, 'auc_roc': 0.7282436588914626, 'train_size': 119145, 'test_size': 29787}
Classification report:
              precision    recall  f1-score   support

           0     0.6739    0.6214    0.6466     14894
           1     0.6487    0.6993    0.6731     14893

    accuracy                         0.6604     29787
   macro avg     0.6613    0.6604    0.6598     29787
weighted avg     0.6613    0.6604    0.6598     29787


=== Tuning RANDOM_FOREST ===
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params: {'n_estimators': 500, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 2

In [ ]:
# Save selected best model for Django API
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print('Selected model saved:', best_model_name)
print('Model saved to:', MODEL_PATH)

Selected model saved: random_forest
Model saved to: C:\Users\dali\EduStat-TN\backend\orientation\ml\model_main.joblib


: 

## Next Step

After saving the model, restart Django server if needed and test `POST /api/predict/` with real examples.
